# 05 · Silver job + grade
Reference data with SCD2 (module 06 §4).

✅ Expect hcmjob **60** · hcmgrade **12** · GradeLevel 1–12.

In [ ]:
from pyspark.sql import functions as F
from hcm_utils import to_int, latest_batch, scd2_merge

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("run_id", "manual")
catalog = dbutils.widgets.get("catalog")
run_id  = dbutils.widgets.get("run_id")

In [ ]:
jobs = (latest_batch(f"{catalog}.bronze.per_jobs")
        .select(to_int("JOB_ID").alias("JobId"),
                F.trim("NAME").alias("JobName"),
                F.trim("JOB_FAMILY").alias("JobFamily"),
                F.lit(run_id).alias("RunId"))
        .dropDuplicates(["JobId"]))
scd2_merge(jobs, f"{catalog}.silver.hcmjob", key="JobId", tracked_cols=["JobName", "JobFamily"])

grades = (latest_batch(f"{catalog}.bronze.per_grades_f")
          .select(to_int("GRADE_ID").alias("GradeId"),
                  F.trim("NAME").alias("GradeName"),
                  F.regexp_extract("NAME", r"^G(\d+)", 1).cast("int").alias("GradeLevel"),
                  F.lit(run_id).alias("RunId"))
          .dropDuplicates(["GradeId"]))
scd2_merge(grades, f"{catalog}.silver.hcmgrade", key="GradeId", tracked_cols=["GradeName", "GradeLevel"])

print("hcmjob:", spark.table(f"{catalog}.silver.hcmjob").filter("IsCurrent").count(),
      "| hcmgrade:", spark.table(f"{catalog}.silver.hcmgrade").filter("IsCurrent").count())